# Chapter 11：Attention 基础

这是 attention/FlashAttention 进阶部分的起点。本章不写 Triton kernel，先理解 q/k/v `[B,H,S,D]`、scores、causal mask、softmax 和输出。

In [ ]:
from pathlib import Path
import math
import sys

ROOT = Path.cwd()
if ROOT.name.startswith("chapter_"):
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import torch.nn.functional as F
import triton
import triton.language as tl

from common.benchmark import bench
from common.check import assert_close
from common.utils import get_device, set_seed

device = get_device()
set_seed(0)

## 输入与公式

`scores = q @ k.T / sqrt(D)` 得到 `[B,H,S,S]`；softmax 后与 v 相乘回到 `[B,H,S,D]`。

In [ ]:
def _validate_qkv(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> None:
    if q.ndim != 4 or q.shape != k.shape or q.shape != v.shape:
        raise ValueError("q, k, and v must share shape [B, H, S, D]")
    if not q.is_cuda or not k.is_cuda or not v.is_cuda:
        raise ValueError("q, k, and v must be CUDA tensors")
    if q.dtype != k.dtype or q.dtype != v.dtype:
        raise ValueError("q, k, and v must share a dtype")


def torch_attention_reference(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, causal: bool = False) -> torch.Tensor:
    _validate_qkv(q, k, v)
    scale = 1.0 / math.sqrt(q.shape[-1])
    scores = torch.matmul(q.float(), k.float().transpose(-2, -1)) * scale
    if causal:
        query_positions = torch.arange(q.shape[-2], device=q.device)[:, None]
        key_positions = torch.arange(k.shape[-2], device=k.device)[None, :]
        scores = scores.masked_fill(key_positions > query_positions, -float("inf"))
    probabilities = torch.softmax(scores, dim=-1)
    return torch.matmul(probabilities, v.float()).to(q.dtype)

## Causal mask

标准 causal self-attention 只屏蔽 `key_index > query_index`，因此每个 query 仍能看到自己。

In [ ]:
q = torch.randn(2,4,128,64,device=device,dtype=torch.float16)
k = torch.randn_like(q)
v = torch.randn_like(q)
print(torch_attention_reference(q,k,v,False).shape)
print(torch_attention_reference(q,k,v,True).shape)

## 与 PyTorch SDPA 比较

如果当前 PyTorch 提供 SDPA，就把显式公式与官方算子比较；这里没有调用 FlashAttention 或 Triton attention。

In [ ]:
def maybe_compare_with_torch_sdpa(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, causal: bool = False) -> None:
    if not hasattr(F, "scaled_dot_product_attention"):
        print("scaled_dot_product_attention is unavailable in this PyTorch version")
        return
    expected = F.scaled_dot_product_attention(q, k, v, is_causal=causal)
    actual = torch_attention_reference(q, k, v, causal)
    assert_close(f"attention vs SDPA causal={causal}", actual.float(), expected.float(), rtol=1e-2, atol=1e-2)

maybe_compare_with_torch_sdpa(q,k,v,False)
maybe_compare_with_torch_sdpa(q,k,v,True)

## Benchmark

显式实现会 materialize `[S,S]` scores，是下一章要解决的内存问题。

In [ ]:
print(f'explicit attention={bench(lambda: torch_attention_reference(q,k,v)):.3f} ms')

## 小结与练习

练习：打印 scores 的 shape 和元素数，并把 S 从 128 增加到 256，观察其按 S² 增长。